# Stage 0 — Cascade Evaluation on a Real Model (Qwen3-4B-Thinking-2507)

This notebook runs the Stage 0 cascade-evaluation harness (`quant_research.cascade`) against a real
model. Everything up to the first run of this notebook (`cascade.py`, `hf_runner.py`, both test
suites) was validated on synthetic/mock data only.

**What this does:**
1. Loads a small subset of [MATH-500](https://huggingface.co/datasets/HuggingFaceH4/MATH-500) problems.
2. Generates a reference trace with **Qwen3-4B-Thinking-2507 in BF16**.
3. Teacher-forces that trace's tokens through an **NF4-quantized** copy of the same model (via
   `bitsandbytes`), and — as a sanity check on the harness itself — through the **BF16 model
   teacher-forced on its own trace**.
4. Also free-runs the NF4 candidate independently, so `propagated_to_final_answer` reflects the
   candidate's *actual* answer rather than an assumption.
5. Scores everything with `first_error_step` / `error_propagation_rate` /
   `divergence_position_distribution` from `cascade.py`.

**Fixed since the first run (2026-08-08):** that run's self-check — the BF16 model teacher-forced on
its *own* trace — diverged on 8/8 problems, and every trace hit the token cap before finishing. Two
fixes:
- `load_model` now defaults to `attn_implementation="eager"`. The likely cause of the self-check
  failure was `generate()`'s incremental KV-cached decoding and the harness's single-shot bulk
  forward pass dispatching to different fused-attention kernels under sdpa/flash-attention, which can
  diverge slightly in bf16 and flip an argmax at a close call. Eager attention is plain matmuls with
  no cache-shape-dependent fused kernel, so both code paths compute the same thing. It's slower per
  token — an accepted tradeoff for correctness here.
- `MAX_NEW_TOKENS` raised from 1024 to 4096, and `generate_reference` now reports whether each trace
  was truncated rather than reaching EOS naturally, logged per-problem and aggregated below. A
  truncated trace's extracted answer isn't trustworthy — the model never got to state one.

**Scope / honesty check:** NF4 (bitsandbytes) is a calibration-free PTQ stand-in for the eventual
MR-GPTQ / NVFP4 conditions EXPERIMENTS.md's Stage 1 actually wants — Colab GPUs (T4/A100,
Ampere-class or older) have no native FP4 tensor cores, so this cannot produce a real NVFP4 result or
a like-for-like Stage 1 attribution comparison. This run is a **harness correctness check against a
real model**: does the plumbing work end-to-end and do the metrics come out sane. N is small (a
handful of problems) for the same reason — validating the pipeline, not statistical power.

**Requirements:** Colab GPU runtime (Runtime → Change runtime type → GPU). Eager attention + 4096
tokens is meaningfully slower than the first run — an A100 runtime is recommended over a free-tier T4
this time; if you're stuck on T4, expect this to take a while, and consider dropping `N_PROBLEMS`
first rather than `MAX_NEW_TOKENS` (a short token budget is what broke the first run's results).


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
import os

# Anchor to an absolute path and always start from a clean clone. Checking
# isdir("quant-research") relative to the *current* directory is not safe to
# re-run: if a prior cell execution in this same kernel already left cwd
# inside quant-research (e.g. "Run all" triggered twice without a full
# restart), the relative check passes trivially and this clones a second,
# nested copy at quant-research/quant-research — whichever commit that
# nested clone happened to be cloned at is then what actually gets imported,
# which can silently be stale. Removing any existing checkout first makes
# this cell idempotent regardless of what state the kernel was already in.
os.chdir("/content")
if os.path.isdir("quant-research"):
    !rm -rf quant-research
!git clone https://github.com/ZGCompute/quant-research.git
%cd /content/quant-research

# Use the %pip magic, not !pip — on Colab, !pip can install into a
# different environment than the one the notebook kernel is actually
# running (a long-standing Colab quirk), which silently produces exactly
# "ModuleNotFoundError: No module named 'quant_research'" in the next
# cell even though pip reports success here. %pip always targets the
# running kernel's environment.
%pip install -q -e ".[hf]"

# A second, distinct gotcha: editable installs register an import hook
# via a .pth file in site-packages, but .pth files are only processed by
# site.py at interpreter startup — a live Colab kernel already passed
# that point, so `import quant_research` can still fail right after a
# successful install even with %pip. Put src/ on sys.path directly so
# the import works in this same cell without needing a kernel restart.
import sys
from pathlib import Path

src_path = str(Path.cwd() / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import importlib

import quant_research

importlib.reload(quant_research)
print("quant_research loaded from:", quant_research.__file__)


In [ ]:
import json
from pathlib import Path

from datasets import load_dataset

from quant_research.cascade import (
    divergence_position_distribution,
    error_propagation_rate,
    score_cascade,
)
from quant_research.hf_runner import (
    extract_boxed_answer,
    generate_reference,
    load_model,
    teacher_forced_step_distributions,
    unload_model,
)

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
N_PROBLEMS = 8
MAX_NEW_TOKENS = 4096


## Load a small MATH-500 subset

`answer` is the dataset's ground-truth answer, used only to report whether the reference model
itself got the problem right (not part of the cascade metric, which compares candidate-vs-reference,
not candidate-vs-ground-truth).


In [ ]:
dataset = load_dataset("HuggingFaceH4/MATH-500", split="test")
problems = dataset.select(range(N_PROBLEMS))
for p in problems:
    print(p["unique_id"], "-", p["problem"][:80].replace("\n", " "), "...")


## Load the reference model

Loaded one at a time rather than both BF16 and NF4 resident together — running both
simultaneously left too little headroom for eager attention's O(n^2) activation memory during
the teacher-forced pass and caused a CUDA OOM on Stage 0's second real run. Phase 1 below runs
everything that needs the reference model; the candidate is loaded only after the reference is
unloaded (see the phase break further down).


In [ ]:
reference = load_model(MODEL_ID, quant_mode="bf16")


## Build a thinking-mode prompt


In [ ]:
def build_prompt(problem: str) -> str:
    messages = [{
        "role": "user",
        "content": problem + "\n\nPlease reason step by step, and put your final answer within \\boxed{}.",
    }]
    return reference.tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=True
    )


## Run Stage 0

Two phases, so only one 4B-class model is ever resident on the GPU at once (see the note above
on why): 

**Phase 1 — reference only, for every problem:**
1. Generate the reference (BF16) trace freely, tracking whether it was truncated at
   `MAX_NEW_TOKENS` rather than reaching EOS naturally.
2. Teacher-force the reference model on its *own* trace as a harness sanity check — with eager
   attention this should now diverge close to never; if it still diverges often, treat that as a
   remaining harness bug, not a quantization effect.

**Phase 2 — after `reference` is unloaded and `candidate` is loaded, for every problem:**

3. Teacher-force the NF4 candidate on the reference trace to find where its own next-token
   distribution first diverges from the reference (`first_error_step`).
4. Free-run the NF4 candidate independently (also tracking truncation) to get its actual final
   answer, so `propagated_to_final_answer` reflects a real outcome rather than an assumption.


In [ ]:
# Phase 1: reference-only pass over every problem.
reference_phase = []

for problem in problems:
    prompt = build_prompt(problem["problem"])

    ref_token_ids, ref_text, ref_truncated = generate_reference(
        reference, prompt, max_new_tokens=MAX_NEW_TOKENS
    )
    ref_answer = extract_boxed_answer(ref_text)

    self_check_steps = teacher_forced_step_distributions(reference, prompt, ref_token_ids)
    self_check_result = score_cascade(ref_token_ids, self_check_steps, ref_answer, ref_answer)

    reference_phase.append({
        "problem": problem,
        "prompt": prompt,
        "ref_token_ids": ref_token_ids,
        "ref_truncated": ref_truncated,
        "ref_answer": ref_answer,
        "self_check_result": self_check_result,
    })

    print(
        f"{problem['unique_id']}: trace_len={len(ref_token_ids)} "
        f"ref_truncated={ref_truncated} self_check_diverged={self_check_result.diverged}"
    )


Free the reference model before loading the candidate — this is the point of the two-phase
split, so don't skip it.


In [ ]:
unload_model(reference)
candidate = load_model(MODEL_ID, quant_mode="nf4")


In [ ]:
# Phase 2: candidate-only pass over every problem, scored against Phase 1's reference traces.
records = []
cascade_results = []
trace_lengths = []

for entry in reference_phase:
    problem = entry["problem"]
    prompt = entry["prompt"]
    ref_token_ids = entry["ref_token_ids"]
    ref_answer = entry["ref_answer"]
    self_check_result = entry["self_check_result"]

    candidate_steps = teacher_forced_step_distributions(candidate, prompt, ref_token_ids)

    cand_token_ids, cand_text, cand_truncated = generate_reference(
        candidate, prompt, max_new_tokens=MAX_NEW_TOKENS
    )
    cand_answer = extract_boxed_answer(cand_text)

    candidate_result = score_cascade(ref_token_ids, candidate_steps, ref_answer, cand_answer)

    cascade_results.append(candidate_result)
    trace_lengths.append(len(ref_token_ids))
    records.append({
        "unique_id": problem["unique_id"],
        "level": problem["level"],
        "trace_length": len(ref_token_ids),
        "reference_truncated": entry["ref_truncated"],
        "candidate_truncated": cand_truncated,
        "reference_answer": ref_answer,
        "dataset_answer": problem["answer"],
        "reference_correct": ref_answer == problem["answer"],
        "self_check_diverged": self_check_result.diverged,
        "self_check_first_error_step": self_check_result.first_error_step,
        "self_check_divergence_margin": self_check_result.divergence_margin,
        "self_check_divergence_entropy": self_check_result.divergence_entropy,
        "candidate_answer": cand_answer,
        "candidate_diverged": candidate_result.diverged,
        "candidate_first_error_step": candidate_result.first_error_step,
        "candidate_divergence_margin": candidate_result.divergence_margin,
        "candidate_divergence_entropy": candidate_result.divergence_entropy,
        "candidate_propagated_to_final_answer": candidate_result.propagated_to_final_answer,
    })

    print(
        f"{problem['unique_id']}: trace_len={len(ref_token_ids)} "
        f"cand_truncated={cand_truncated} "
        f"candidate_diverged={candidate_result.diverged} "
        f"first_error_step={candidate_result.first_error_step} "
        f"propagated={candidate_result.propagated_to_final_answer}"
    )


## Aggregate and save


In [ ]:
self_check_divergence_rate = sum(r["self_check_diverged"] for r in records) / len(records)
reference_truncation_rate = sum(r["reference_truncated"] for r in records) / len(records)
candidate_truncation_rate = sum(r["candidate_truncated"] for r in records) / len(records)

print(f"self-check divergence rate (should be ~0): {self_check_divergence_rate:.3f}")
print(f"reference truncation rate (should be ~0): {reference_truncation_rate:.3f}")
print(f"candidate truncation rate (should be ~0): {candidate_truncation_rate:.3f}")
print(f"candidate error-propagation rate: {error_propagation_rate(cascade_results):.3f}")

positions = divergence_position_distribution(cascade_results, trace_lengths)
print(f"candidate divergence positions (0=start of trace, 1=end): {positions}")

if self_check_divergence_rate > 0.2:
    print(
        "\nWARNING: self-check divergence rate is still high. The eager-attention fix may not be "
        "enough on its own — don't trust candidate_* numbers below until this is resolved."
    )
if max(reference_truncation_rate, candidate_truncation_rate) > 0.2:
    print(
        "\nWARNING: truncation rate is still high. Raise MAX_NEW_TOKENS further before trusting "
        "reference_correct / candidate_answer / propagated_to_final_answer."
    )


In [ ]:
output_path = Path("stage0_results.json")
output_path.write_text(json.dumps({
    "model_id": MODEL_ID,
    "n_problems": N_PROBLEMS,
    "records": records,
    "self_check_divergence_rate": self_check_divergence_rate,
    "error_propagation_rate": error_propagation_rate(cascade_results),
    "divergence_positions": positions,
}, indent=2))
print(f"Saved to {output_path.resolve()}")

# Optional: persist to Drive so results survive the Colab session ending.
# from google.colab import drive
# drive.mount('/content/drive')
# output_path.rename('/content/drive/MyDrive/stage0_results.json')


## Next steps

- Check the two warnings (if any) printed above before trusting anything else in this notebook. If
  self-check divergence or truncation rate are still high, that's a harness issue to chase down, not
  a quantization result.
- Once both rates are low, scale `N_PROBLEMS` up and add AIME 2025/2026 + GPQA-Diamond, per
  EXPERIMENTS.md Stage 0's full benchmark list, not just MATH-500.
- Stage 1 (error attribution: weight vs. KV-cache vs. activation) needs a genuinely like-for-like
  comparison — bitsandbytes NF4 here is not that; it conflates weight quantization with
  bitsandbytes-specific compute-path effects. Stage 1 will need MR-GPTQ/NVFP4-style conditions, which
  needs Blackwell-class hardware not available on Colab (see the hardware roadmap in project memory).
